# Stage 04 — Scalar Temperature Calibration

## Stage Overview
This stage consumes only locked Stage 03 artifacts, generates CLEAN DEV logits, and fits one scalar temperature per model/seed by DEV NLL minimization.

## Scientific Objective
H2 evaluates transport of source-trained calibration downstream. Stage 04 performs calibration fitting only; BanglaBait TEST and external datasets are not read.


## Input Contract

### Required Input
One Kaggle Dataset containing the frozen Swarabyanjan external-data package may remain mounted; Stage 04 does not read external labels.

### Runtime
Kaggle Notebook.

### Accelerator
**GPU optional but recommended** for transformer DEV inference. CPU execution is scientifically valid but slower.

### Pipeline Position
03 / 04 / 05 / 06

### Upstream Requirements
A completed Stage 03 `PASS` artifact root, mounted as a prior Kaggle notebook output or available under `/kaggle/working`.

### Read-only Inputs
Stage 03 clean DEV and locked model artifacts are read-only.

### Generated Outputs
`/kaggle/working/03_results/04_calibration/`, including temperatures, DEV predictions, calibration report, manifests, hashes, and runtime metadata.

### Manual Uploads
No manually generated intermediate files are required. Prior-stage outputs should be mounted directly as Kaggle notebook outputs.


## Environment Validation
The existing Kaggle compiled stack is preserved. Missing application packages are installed without upgrading NumPy/SciPy/pandas/scikit-learn/pyarrow/PyTorch, followed by a fresh-process import/API probe.


In [1]:
# Environment preflight — standard library only.
# The compiled Kaggle scientific stack is preserved. Only missing application-level packages may be installed.
from pathlib import Path
import importlib.metadata as _md
import importlib.util as _util
import json as _json
import os as _os
import subprocess as _subprocess
import sys as _sys
import tempfile as _tempfile
import textwrap as _textwrap

STAGE_ENVIRONMENT_ID = "SWARABYANJAN-KAGGLE-ENV-v2.0.0"
CORE_DISTRIBUTIONS = {
    "numpy":"numpy",
    "scipy":"scipy",
    "pandas":"pandas",
    "scikit-learn":"sklearn",
    "pyarrow":"pyarrow",
    "torch":"torch",
    "statsmodels":"statsmodels",
    "joblib":"joblib",
}
APPLICATION_REQUIREMENTS = [('interpret-core', 'interpret', 'interpret-core>=0.6,<0.8'), ('transformers', 'transformers', 'transformers>=4.45,<6'), ('sentencepiece', 'sentencepiece', 'sentencepiece>=0.1.99,<0.3')]

def _dist_version(name):
    try:
        return _md.version(name)
    except _md.PackageNotFoundError:
        return None

def _module_present(name):
    try:
        return _util.find_spec(name) is not None
    except Exception:
        return False

initial_versions = {name:_dist_version(name) for name in list(CORE_DISTRIBUTIONS)+[x[0] for x in APPLICATION_REQUIREMENTS]}
missing_core = [dist for dist,module in CORE_DISTRIBUTIONS.items() if not _module_present(module)]
if missing_core:
    raise RuntimeError(
        "WHAT FAILED: the standard Kaggle compiled scientific stack is incomplete. "
        f"Missing core distributions/modules: {missing_core}. "
        "WHY: installing or replacing NumPy/SciPy/pandas/scikit-learn/pyarrow/torch inside this notebook can create ABI conflicts. "
        "SAFE TO CONTINUE: NO. Start a fresh standard Kaggle Python runtime; do not upgrade the scientific stack."
    )

missing_apps = [(dist,module,spec) for dist,module,spec in APPLICATION_REQUIREMENTS if not _module_present(module)]
installed_now = []
if missing_apps:
    # Pin every existing compiled/core package so the dependency resolver cannot replace Kaggle's ABI-compatible stack.
    constraints = []
    for dist in CORE_DISTRIBUTIONS:
        v = _dist_version(dist)
        if v:
            constraints.append(f"{dist}=={v}")
    cfile = Path(_tempfile.gettempdir()) / f"swarabyanjan_04_constraints.txt"
    cfile.write_text("\n".join(constraints)+"\n", encoding="utf-8")
    specs = [spec for _,_,spec in missing_apps]
    cmd = [
        _sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "--no-input", "--prefer-binary",
        "--constraint", str(cfile), *specs
    ]
    proc = _subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(
            "WHAT FAILED: installation of missing application-level dependencies failed. "
            f"Requested: {specs}. "
            "WHY: pip could not satisfy them without changing the pinned Kaggle scientific stack or network/package access failed. "
            "SAFE TO CONTINUE: NO.\n"
            f"pip stdout:\n{proc.stdout[-4000:]}\n"
            f"pip stderr:\n{proc.stderr[-4000:]}"
        )
    installed_now = specs

# Import/API validation occurs in a fresh subprocess before compiled packages are imported into this notebook kernel.
_probe = r"""
import json
import numpy, scipy, pandas, sklearn, pyarrow, torch, statsmodels, joblib
from scipy.optimize import minimize_scalar
from interpret.glassbox import ExplainableBoostingClassifier
from transformers import AutoTokenizer, AutoModelForSequenceClassification
print(json.dumps({
 "numpy":numpy.__version__,"scipy":scipy.__version__,"pandas":pandas.__version__,
 "sklearn":sklearn.__version__,"pyarrow":pyarrow.__version__,"torch":torch.__version__,
 "statsmodels":statsmodels.__version__
}, sort_keys=True))
"""
probe_proc = _subprocess.run([_sys.executable, "-c", _probe], capture_output=True, text=True)
if probe_proc.returncode != 0:
    raise RuntimeError(
        "WHAT FAILED: post-setup import/API validation failed. "
        "WHY: the runtime is still incompatible or a required API is unavailable. "
        "SAFE TO CONTINUE: NO. Restart with a fresh Kaggle Python runtime rather than upgrading the scientific stack.\n"
        f"probe stdout:\n{probe_proc.stdout[-4000:]}\n"
        f"probe stderr:\n{probe_proc.stderr[-4000:]}"
    )

ENVIRONMENT_PREFLIGHT = {
    "schema_version": 1,
    "environment_id": STAGE_ENVIRONMENT_ID,
    "stage_id": "04",
    "policy": "preserve Kaggle compiled scientific stack; install only missing application-level packages",
    "accelerator_note": 'GPU optional; recommended for transformer DEV inference',
    "initial_versions": initial_versions,
    "installed_now": installed_now,
    "validated_versions": {name:_dist_version(name) for name in list(CORE_DISTRIBUTIONS)+[x[0] for x in APPLICATION_REQUIREMENTS]},
    "probe_stdout": probe_proc.stdout.strip(),
}
print("Environment validation: PASS")
print(probe_proc.stdout.strip())


Environment validation: PASS
{"numpy": "2.0.2", "pandas": "2.3.3", "pyarrow": "24.0.0", "scipy": "1.16.3", "sklearn": "1.6.1", "statsmodels": "0.14.6", "torch": "2.10.0+cu128"}


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.metadata as im, json, math, os, platform, re, subprocess, sys
import numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
import statsmodels.api as sm
from sklearn.metrics import log_loss, brier_score_loss
import joblib

STAGE_ID="04"; STAGE_NAME="04_calibration"; PROTOCOL_VERSION="SWARABYANJAN-CROSSPLATFORM-v1.0.0"
EXPECTED_ZIP_SHA256="e0de5c224f406fe22d8ad30804217465f789c49c70e1d3afbe00afd28deb7ccf"
SEEDS=[42,137,2718,31415,65537]; MODEL_KEYS=["ebm","tfidf_lr","banglabert","xlmr"]
WORK_ROOT=Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
SEARCH_ROOTS=[Path("/kaggle/input"),WORK_ROOT,Path.cwd()] if Path("/kaggle/input").exists() else [WORK_ROOT,Path.cwd()]
OUT_ROOT=WORK_ROOT/"03_results"/STAGE_NAME
if OUT_ROOT.exists() and any(OUT_ROOT.iterdir()): raise RuntimeError(f"Refusing overwrite: {OUT_ROOT}")
OUT_ROOT.mkdir(parents=True,exist_ok=True)

def utc_now(): return datetime.now(timezone.utc).isoformat()
def cjb(o): return json.dumps(o,ensure_ascii=False,sort_keys=True,separators=(",",":")).encode("utf-8")
def sha256_file(p,chunk=8*1024*1024):
    h=hashlib.sha256(); f=open(p,"rb")
    while True:
        b=f.read(chunk)
        if not b: f.close(); return h.hexdigest()
        h.update(b)
def write_json(p,o): Path(p).write_text(json.dumps(o,ensure_ascii=False,sort_keys=True,indent=2)+"\n",encoding="utf-8",newline="\n")
def write_text(p,t): Path(p).write_text(t,encoding="utf-8",newline="\n")
def set_status(s,m,d=None):
    write_json(OUT_ROOT/"STAGE_STATUS.json",{"stage_id":STAGE_ID,"status":s,"message":m,"timestamp_utc":utc_now(),"details":d or {}}); print(f"STAGE 04 STATUS: {s} — {m}")
def hard_stop(m,d=None): set_status("HARD_STOP",m,d); raise RuntimeError(m)
def pkg(n):
    try:return im.version(n)
    except:return None
def git_state():
    try:
        head=subprocess.check_output(["git","rev-parse","HEAD"],stderr=subprocess.DEVNULL,text=True).strip()
        status=subprocess.check_output(["git","status","--porcelain"],stderr=subprocess.DEVNULL,text=True)
        return {"available":True,"head":head,"dirty":bool(status.strip()),"status_porcelain":status[:20000]}
    except Exception:
        return {"available":False,"head":None,"dirty":None,"status_porcelain":None}
write_json(OUT_ROOT/"ENVIRONMENT_PREFLIGHT.json",ENVIRONMENT_PREFLIGHT)
set_status("RUNNING","Stage-04 initialization complete")


STAGE 04 STATUS: RUNNING — Stage-04 initialization complete


## Upstream Artifact Verification
Exactly one unique Stage 03 content identity is accepted. Duplicate physical mounts of identical bytes are collapsed deterministically; distinct upstream identities hard-stop.


In [3]:
# ---------- Locate and cryptographically verify exactly one Stage-03 artifact root ----------
def parse_sumfile(path):
    out={}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        h,rel=line.split(None,1); rel=rel.lstrip("*").strip().replace("\\","/")
        if not re.fullmatch(r"[0-9a-f]{64}",h.lower()): hard_stop("Malformed upstream checksum line",{"line":line})
        pp=Path(rel)
        if pp.is_absolute() or ".." in pp.parts: hard_stop("Unsafe upstream checksum path",{"path":rel})
        if rel in out: hard_stop("Duplicate upstream checksum path",{"path":rel})
        out[rel]=h.lower()
    return out

def find_stage03_root():
    c=[]
    for root in SEARCH_ROOTS:
        if not root.exists(): continue
        for p in root.rglob("MODEL_ARTIFACT_MANIFEST.json"):
            parent=p.parent
            if (parent/"STAGE03_SHA256SUMS.txt").is_file() and (parent/"D0_REPORT.json").is_file(): c.append(parent.resolve())
    c=sorted(set(c),key=str)
    if not c: hard_stop("Stage-03 artifact root was not found",{"safe_to_continue":False})
    identities={}
    for x in c:
        identities.setdefault(sha256_file(x/"STAGE03_SHA256SUMS.txt"),[]).append(x)
    if len(identities)!=1:
        hard_stop("Multiple distinct Stage-03 artifact identities were found",{"candidates":[str(x) for x in c],"safe_to_continue":False})
    chosen=sorted(next(iter(identities.values())),key=str)[0]
    if len(c)>1: print("Upstream discovery: identical Stage-03 mounts collapsed; using",chosen)
    return chosen
STAGE03_ROOT=find_stage03_root(); sums=parse_sumfile(STAGE03_ROOT/"STAGE03_SHA256SUMS.txt")
fail=[]
for rel,h in sorted(sums.items()):
    p=STAGE03_ROOT/rel
    if not p.is_file(): fail.append({"path":rel,"reason":"missing"})
    elif sha256_file(p)!=h: fail.append({"path":rel,"reason":"sha256_mismatch"})
if fail: hard_stop("Stage-03 artifact verification failed",{"failures":fail[:20],"count":len(fail)})
D0=json.loads((STAGE03_ROOT/"D0_REPORT.json").read_text(encoding="utf-8"))
if D0.get("status")!="PASS": hard_stop("Stage-03 D0 was not PASS")
MODEL_MANIFEST=json.loads((STAGE03_ROOT/"MODEL_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
CONFIG=json.loads((STAGE03_ROOT/"MODEL_CONFIG_LOCK.json").read_text(encoding="utf-8"))
CUE_SCHEMA=json.loads((STAGE03_ROOT/"CUE_FEATURE_SCHEMA.json").read_text(encoding="utf-8"))
if MODEL_MANIFEST.get("frozen_package_sha256")!=EXPECTED_ZIP_SHA256: hard_stop("Stage-03 frozen package identity mismatch")
if sorted(MODEL_MANIFEST.get("seeds",[]))!=sorted(SEEDS): hard_stop("Stage-03 seed contract mismatch")
if sorted({r["model"] for r in MODEL_MANIFEST["models"]})!=sorted(MODEL_KEYS): hard_stop("Stage-03 model panel mismatch")
if CONFIG.get("config_lock_sha256")!=MODEL_MANIFEST.get("model_config_lock_sha256"): hard_stop("Stage-03 model-config lock mismatch")
if CUE_SCHEMA.get("cue_schema_sha256")!=MODEL_MANIFEST.get("cue_schema_sha256"): hard_stop("Stage-03 cue-schema lock mismatch")
CLEAN_DEV=STAGE03_ROOT/MODEL_MANIFEST["clean_split_artifacts"]["dev"]["path"]
if sha256_file(CLEAN_DEV)!=MODEL_MANIFEST["clean_split_artifacts"]["dev"]["sha256"]: hard_stop("Clean DEV artifact hash mismatch")

# Serialized EBM and scikit-learn artifacts are safest under the exact application/library versions used in Stage 03.
S3_RUN=json.loads((STAGE03_ROOT/"RUN_METADATA.json").read_text(encoding="utf-8"))
s3_pkgs=S3_RUN.get("environment",{}).get("packages",{})
critical_serialization_versions={"scikit-learn":pkg("scikit-learn"),"interpret-core":pkg("interpret-core"),"joblib":pkg("joblib")}
version_mismatch={}
for name,current in critical_serialization_versions.items():
    expected=s3_pkgs.get(name)
    if expected and current and expected!=current:version_mismatch[name]={"stage03":expected,"current":current}
if version_mismatch:
    hard_stop("Critical serialized-model library versions differ from Stage 03",
              {"mismatches":version_mismatch,"safe_to_continue":False,
               "action":"Run Stage 04 on the same Kaggle base image/package set as Stage 03; do not upgrade the compiled stack."})
print("Stage-03 upstream contract: PASS",STAGE03_ROOT)


Stage-03 upstream contract: PASS /kaggle/input/notebooks/smalakarishere/03-primary-model-development/03_results/03_primary_model_development


## Cue Extraction and DEV Inference
Cue extraction is reused from the locked Stage 03 schema. Transformer inference may reduce physical inference batch size after CUDA OOM because inference batching does not alter the scientific estimand.


In [4]:
# ---------- Shared cue extraction from the locked Stage-03 schema ----------
import unicodedata

def norm_text(v):
    if v is None or (isinstance(v,float) and math.isnan(v)): return ""
    t=unicodedata.normalize("NFKC",str(v)); t=t.replace("\u200b","").replace("\u200c","").replace("\u200d","").replace("\ufeff","")
    return re.sub(r"\s+"," ",t.casefold().strip())

def term_count(text,terms):
    toks=text.split(); from collections import Counter
    cc=Counter(toks); return sum(text.count(norm_text(t)) if " " in norm_text(t) else cc.get(norm_text(t),0) for t in terms)
def cue_features(titles):
    rows=[]
    for raw in titles:
        text=norm_text(raw); toks=text.split(); row={}
        for name,spec in CUE_SCHEMA["families"].items():
            if name=="specificity_numerals": c=sum(len(re.findall(p,text,flags=re.I)) for p in spec["patterns"])
            elif name=="punctuation_intensity": c=sum(text.count(p) for p in spec["punctuation"])+len(re.findall(spec["repeated_pattern"],text))
            else: c=term_count(text,spec.get("terms",[]))+sum(text.count(p) for p in spec.get("punctuation",[]))
            row[name]=int(c)
        chars=len(text); latin=sum("a"<=ch<="z" for ch in text)
        row.update({"token_count":len(toks),"character_count":chars,"mean_token_length":float(np.mean([len(x) for x in toks])) if toks else 0.0,
                    "latin_code_mixing_proportion":latin/chars if chars else 0.0})
        rows.append(row)
    return pd.DataFrame(rows,columns=list(CUE_SCHEMA["families"])+CUE_SCHEMA["structural_controls"])


In [5]:
# ---------- Generate DEV logits from locked Stage-03 models only ----------
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

dev=pd.read_parquet(CLEAN_DEV).sort_values("source_row_id",kind="stable").reset_index(drop=True)
if set(dev["label"].unique())!={0,1}: hard_stop("Clean DEV labels are not binary {0,1}")

class InferDS(Dataset):
    def __init__(self,texts,tok,max_length): self.enc=tok(list(texts),truncation=True,padding=False,max_length=max_length)
    def __len__(self): return len(next(iter(self.enc.values())))
    def __getitem__(self,i): return {k:torch.tensor(v[i],dtype=torch.long) for k,v in self.enc.items()}
def infer_collate(batch,tok): return tok.pad(batch,padding=True,return_tensors="pt")
def transformer_logits(path,texts,max_length,batch_size=64):
    tok=AutoTokenizer.from_pretrained(path,local_files_only=True)
    model=AutoModelForSequenceClassification.from_pretrained(path,local_files_only=True)
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); model.to(device); model.eval()
    ds=InferDS(texts,tok,max_length)
    last_error=None
    for bs in [batch_size,32,16,8]:
        bs=min(bs,batch_size)
        if bs<=0: continue
        try:
            loader=DataLoader(ds,batch_size=bs,shuffle=False,collate_fn=lambda b:infer_collate(b,tok))
            z=[]
            with torch.no_grad():
                for batch in loader:
                    batch={k:v.to(device) for k,v in batch.items()}
                    lo=model(**batch).logits.detach().cpu().numpy()
                    z.extend((lo[:,1]-lo[:,0]).tolist())
            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            return np.asarray(z,float)
        except RuntimeError as e:
            if "out of memory" not in str(e).lower(): raise
            last_error=e
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    raise RuntimeError(f"WHAT FAILED: transformer inference exhausted safe batch fallbacks. WHY: CUDA memory was insufficient. SAFE TO CONTINUE: NO. Last error: {last_error}")

def model_record(model,seed):
    rec=[r for r in MODEL_MANIFEST["models"] if r["model"]==model and int(r["seed"])==int(seed)]
    if len(rec)!=1: hard_stop("Model artifact record ambiguity",{"model":model,"seed":seed,"count":len(rec)})
    return rec[0]

rows=[]; cues=cue_features(dev["title"]).astype(float)
for model in MODEL_KEYS:
    for seed in SEEDS:
        rec=model_record(model,seed); path=STAGE03_ROOT/rec["artifact_path"]
        # verify state manifest before use
        for f in rec["files"]:
            fp=STAGE03_ROOT/f["path"]
            if not fp.is_file() or sha256_file(fp)!=f["sha256"]: hard_stop("Model-state file hash mismatch",{"path":str(fp)})
        if model=="ebm": z=np.asarray(joblib.load(path).decision_function(cues),float)
        elif model=="tfidf_lr": z=np.asarray(joblib.load(path).decision_function(dev["title"]),float)
        else:
            maxlen=int(CONFIG["models"][model]["base_config"]["max_length"]); z=transformer_logits(path,dev["title"].tolist(),maxlen)
        for i,zz in enumerate(z):
            rows.append({"source_row_id":int(dev.loc[i,"source_row_id"]),"label":int(dev.loc[i,"label"]),"model":model,"seed":seed,"logit":float(zz)})
DEV_LOGITS=pd.DataFrame(rows).sort_values(["model","seed","source_row_id"],kind="stable").reset_index(drop=True)
print("DEV logits generated:",DEV_LOGITS.shape)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

DEV logits generated: (22000, 5)


## Calibration Procedure and Metrics
Scalar temperature scaling is fitted on CLEAN BanglaBait DEV only. No threshold optimization, target recalibration, or external-data tuning is permitted.


In [6]:
# ---------- Scalar temperature fitting and DEV-only calibration report ----------
def nll_from_logits(y,z,T=1.0):
    p=np.clip(expit(np.asarray(z)/float(T)),1e-7,1-1e-7); y=np.asarray(y,int)
    return float(-np.mean(y*np.log(p)+(1-y)*np.log(1-p)))
def fit_temperature(y,z):
    obj=lambda logT:nll_from_logits(y,z,math.exp(float(logT)))
    res=minimize_scalar(obj,bounds=(-5,5),method="bounded",options={"xatol":1e-10})
    if not res.success: hard_stop("Temperature optimization failed",{"message":res.message})
    return float(math.exp(res.x))
def balanced_brier(y,p,w=None):
    y=np.asarray(y,int); p=np.asarray(p,float); vals=[]
    for c in [0,1]:
        m=y==c
        if not m.any(): return None
        err=(p[m]-y[m])**2
        vals.append(float(np.average(err,weights=None if w is None else np.asarray(w)[m])))
    return 0.5*(vals[0]+vals[1])
def calib_intercept_slope(y,p):
    p=np.clip(np.asarray(p,float),1e-7,1-1e-7); x=logit(p); X=sm.add_constant(x,has_constant="add")
    try:
        fit=sm.GLM(np.asarray(y,int),X,family=sm.families.Binomial()).fit()
        return float(fit.params[0]),float(fit.params[1])
    except Exception: return None,None
def metrics(y,p):
    p=np.clip(np.asarray(p,float),1e-7,1-1e-7); y=np.asarray(y,int)
    ci,cs=calib_intercept_slope(y,p)
    return {"nll":float(log_loss(y,p,labels=[0,1])),"brier":float(brier_score_loss(y,p)),"balanced_brier":balanced_brier(y,p),
            "calibration_intercept":ci,"calibration_slope":cs}

temps={}; report=[]; pred_parts=[]
for (model,seed),g in DEV_LOGITS.groupby(["model","seed"],sort=True):
    y=g["label"].to_numpy(); z=g["logit"].to_numpy(); T=fit_temperature(y,z)
    pu=expit(z); pc=expit(z/T)
    temps.setdefault(model,{})[str(int(seed))]=T
    report.append({"model":model,"seed":int(seed),"temperature":T,"uncalibrated":metrics(y,pu),"temperature_scaled":metrics(y,pc),
                   "nll_gain":metrics(y,pu)["nll"]-metrics(y,pc)["nll"]})
    q=g.copy(); q["p_uncalibrated"]=pu; q["temperature"]=T; q["p_calibrated"]=pc; pred_parts.append(q)
DEV_PRED=pd.concat(pred_parts,ignore_index=True).sort_values(["model","seed","source_row_id"],kind="stable")
DEV_PRED.to_parquet(OUT_ROOT/"DEV_PREDICTIONS.parquet",index=False)
write_json(OUT_ROOT/"TEMPERATURES.json",{"schema_version":1,"method":"scalar_temperature_scaling","fit_objective":"DEV NLL",
                                         "source":"clean BanglaBait DEV only","temperatures":temps})
write_json(OUT_ROOT/"CALIBRATION_REPORT.json",{"schema_version":1,"method":"scalar_temperature_scaling","threshold_tuning":False,
                                                "target_recalibration":False,"rows":report})
print("Calibration fitting complete on DEV only")


Calibration fitting complete on DEV only


## Output Artifacts and Stage Decision
Required calibration artifacts are hashed and persisted before `PASS`. TEST and external access remains prohibited in this stage.


In [7]:
# ---------- Stage-04 manifest and PASS ----------
UPSTREAM_HASH=sha256_file(STAGE03_ROOT/"STAGE03_SHA256SUMS.txt")
CAL_MAN={
    "schema_version":1,"stage_id":STAGE_ID,"protocol_version":PROTOCOL_VERSION,
    "upstream_stage03_root":str(STAGE03_ROOT),"upstream_stage03_checksum_manifest_sha256":UPSTREAM_HASH,
    "model_config_lock_sha256":CONFIG["config_lock_sha256"],"model_artifact_manifest_payload_sha256":MODEL_MANIFEST["manifest_payload_sha256"],
    "temperatures_sha256":sha256_file(OUT_ROOT/"TEMPERATURES.json"),"calibration_report_sha256":sha256_file(OUT_ROOT/"CALIBRATION_REPORT.json"),
    "dev_predictions_sha256":sha256_file(OUT_ROOT/"DEV_PREDICTIONS.parquet"),
    "boundary":"DEV only. TEST and external datasets were not read.",
}
CAL_MAN["manifest_payload_sha256"]=hashlib.sha256(cjb(CAL_MAN)).hexdigest(); write_json(OUT_ROOT/"CALIBRATION_ARTIFACT_MANIFEST.json",CAL_MAN)
RUN={"schema_version":1,"stage_id":STAGE_ID,"execution_utc":utc_now(),"python":sys.version,"platform":platform.platform(),
     "packages":{n:pkg(n) for n in ["pandas","numpy","pyarrow","scikit-learn","interpret","transformers","torch","scipy","statsmodels","joblib"]},
     "seeds":SEEDS,"git":git_state(),"input_hashes":{"stage03_checksums":UPSTREAM_HASH},"boundary":{"DEV":"calibration only","TEST":"not accessed","external":"not accessed"}}
write_json(OUT_ROOT/"RUN_METADATA.json",RUN)
write_text(OUT_ROOT/"STAGE04_SUMMARY.md",
           "# Stage 04 execution summary\n\n"
           f"- Calibration method: scalar temperature scaling\n"
           f"- Models × seeds calibrated: {len(report)}\n"
           "- Fitting data: CLEAN BanglaBait DEV only\n"
           "- TEST/external access: none\n")
entries=[]
for p in sorted(OUT_ROOT.rglob("*"),key=str):
    if p.is_file() and p.name not in {"STAGE04_SHA256SUMS.txt","STAGE_STATUS.json"}: entries.append((str(p.relative_to(OUT_ROOT)).replace("\\","/"),sha256_file(p)))
write_text(OUT_ROOT/"STAGE04_SHA256SUMS.txt","\n".join(f"{h}  {r}" for r,h in entries)+"\n")
required=["ENVIRONMENT_PREFLIGHT.json","CALIBRATION_REPORT.json","TEMPERATURES.json","CALIBRATION_ARTIFACT_MANIFEST.json","DEV_PREDICTIONS.parquet","RUN_METADATA.json","STAGE04_SUMMARY.md"]
if any(not (OUT_ROOT/x).is_file() for x in required): hard_stop("Stage-04 required artifact missing")
set_status("PASS","Scalar temperature calibration completed on CLEAN DEV only",{"output_root":str(OUT_ROOT),"checksum_entries":len(entries)})
print("STAGE 04 COMPLETE — PASS")


STAGE 04 STATUS: PASS — Scalar temperature calibration completed on CLEAN DEV only
STAGE 04 COMPLETE — PASS
